# Import Libraries

In [1]:
import pandas as pd
import numpy as np
import spacy
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, Embedding, LSTM, Input, Dropout, GlobalMaxPooling1D, Conv1D, Bidirectional, BatchNormalization, SimpleRNN, Attention, GlobalAveragePooling1D, Bidirectional
from tensorflow.keras.optimizers import Adam
from gensim.models import Word2Vec
from transformers import BertTokenizer, TFBertModel
import tensorflow as tf

pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.max_rows', None)  # Show all rows
pd.set_option('display.max_colwidth', None)  # Show full content in each cell
pd.set_option('display.width', 1000)  # Set max width

# Load spaCy's English model
nlp = spacy.load('en_core_web_sm')

/opt/anaconda3/envs/yt_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def preprocess_text(text):
    # Define interrogative words to KEEP
    interrogatives = {"what", "why", "how", "who", "where", "when", "which", "whom", "whose", "no", "not",
                    "very" ,"too" ,"too" ,"just", "if", "but", "however", "without", "like"}
    custom_stopwords = set(nlp.Defaults.stop_words)
    custom_stopwords -= interrogatives

    doc = nlp(text.lower().strip())  # Lowercase and remove whitespace
    
# Process tokens: lemmatize, filter stopwords/punct/numbers, keep interrogatives
    tokens = [
        token.lemma_ 
        for token in doc 
        if (
            (not token.is_stop or token.text in interrogatives) and  # Keep interrogatives
            not token.is_punct and token.is_alpha                                  # Remove punctuation
            # (token.is_alpha or token.like_num)                       # Keep words/numbers
        )
    ]

    return ' '.join(tokens)

In [3]:
# Tokenize input text
# Load BERT tokenizer and model
model_name = "bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(model_name)
bert_model = TFBertModel.from_pretrained(model_name)

def tokenize_texts(texts, max_len):
    encodings = tokenizer(
        texts.tolist(),
        max_length=max_len,
        truncation=True,
        padding='max_length',
        return_tensors='tf'
    )

    outputs = bert_model(encodings)

    return outputs.last_hidden_state

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['cls.predictions.transform.LayerNorm.bias', 'cls.seq_relationship.bias', 'cls.predictions.bias', 'cls.seq_relationship.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.weight']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFBertModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertModel for predictions w

In [4]:
def make_dataset(texts, labels, tokenizer, shuffle=False, max_len= 20, batch_size = 32):
    def gen():
        for t, l in zip(texts, labels):
            enc = tokenizer(
                t,
                truncation=True,
                padding='max_length',
                max_length=max_len,
                return_tensors='tf'
            )
            yield ({'input_ids': enc['input_ids'][0], 'attention_mask': enc['attention_mask'][0]}, l)

    ds = tf.data.Dataset.from_generator(
        gen,
        output_signature=(
            {'input_ids': tf.TensorSpec(shape=(max_len,), dtype=tf.int32),
             'attention_mask': tf.TensorSpec(shape=(max_len,), dtype=tf.int32)},
            tf.TensorSpec(shape=(), dtype=tf.int32)
        )
    )
    if shuffle:
        ds = ds.shuffle(buffer_size=len(texts))
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

# Pre-Processing

### Import Data

In [77]:
mapping = {
    'knowledge': 'knowledge',
    'remember': 'knowledge',
    'comprehension': 'comprehension',
    'understand': 'comprehension',
    'application': 'application',
    'apply': 'application',
    'analysis': 'analysis',
    'analyse': 'analysis',
    'evaluation': 'evaluation',
    'evaluate': 'evaluation',
    'synthesis': 'synthesis',
    'create': 'synthesis'
}

# Load dataset
df = pd.DataFrame()
for i in [1,2,3,4,5]:
    q_df = pd.read_csv(os.getcwd().replace('notebook' , 'dataset') + '/dataset' + str(i) + '.csv')
    q_df['dataset_id'] = i
    df = pd.concat([df , q_df], )
    
df = df.reset_index(drop=True)

# Apply preprocessing
df['label'] = df['label'].str.lower()
df['label'] = df['label'].replace(mapping)

df['processed_question'] = df['question'].apply(preprocess_text)
df['processed_question'] = [''.join(text) for text in df['processed_question']]

max_len = max(len(tokenizer.encode(text, add_special_tokens=True)) for text in df['question'])
print("Max sequence length:", max_len)

Max sequence length: 95


In [62]:
mapping = {
    'knowledge': 'knowledge',
    'remember': 'knowledge',
    'comprehension': 'comprehension',
    'understand': 'comprehension',
    'application': 'application',
    'apply': 'application',
    'analysis': 'analysis',
    'analyse': 'analysis',
    'evaluation': 'evaluation',
    'evaluate': 'evaluation',
    'synthesis': 'synthesis',
    'create': 'synthesis'
}

# Load dataset
test_df = pd.read_csv(os.getcwd().replace('notebook' , 'dataset') + '/dataset' + str(1) + '.csv')
    
test_df = test_df.reset_index(drop=True)

# Apply preprocessing
test_df['label'] = test_df['label'].str.lower()
test_df['label'] = test_df['label'].replace(mapping)

test_df['processed_question'] = test_df['question'].apply(preprocess_text)
test_df['processed_question'] = [''.join(text) for text in test_df['processed_question']]

## Tokenize

### BERT

In [ ]:
# Parameters
num_classes = 6

# Embedding
x_train = tokenize_texts(df['question'], max_len)
x_test = tokenize_texts(test_df['question'], max_len)

In [ ]:
y_mapper = {
    'knowledge' : 0,
    'comprehension' : 1,
    'application' : 2,
    'analysis' : 3,
    'synthesis' : 4,
    'evaluation' : 5
}

y_mapped = df['label'].map(y_mapper)
y_test_mapped = test_df['label'].map(y_mapper)

y_train = to_categorical(np.asarray(y_mapped))
y_test = to_categorical(np.asarray(y_test_mapped))

# Modelling

## 1D CNN

In [ ]:
cnn_model = Sequential([
    # Input(shape=(max_len, 768)),

    Conv1D(128, 5, activation='gelu', padding= 'same', input_shape=(max_len, 768)),
    BatchNormalization(),
    GlobalMaxPooling1D(),
    Dropout(0.3),

    Dense(64, activation='sigmoid'),
    Dropout(0.4),

    Dense(6, activation='softmax')
])
cnn_model.compile(loss='categorical_crossentropy', optimizer=Adam(learning_rate=1e-4), metrics=['accuracy'])

cnn_model.summary()

/opt/anaconda3/envs/yt_env/lib/python3.10/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_22"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_3 (Conv1D)               │ (None, 84, 128)        │       491,648 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 84, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d_3          │ (None, 128)            │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_71 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_58 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_72 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_59 (Dense)                │ (None, 6)              │           390 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 500,806 (1.91 MB)

 Trainable params: 500,550 (1.91 MB)

 Non-trainable params: 256 (1.00 KB)

In [ ]:
cnn_model.fit(x_train, y_train, epochs=200, batch_size = 32, validation_split= 0.2)

Epoch 1/200
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.1420 - loss: 2.3346 - val_accuracy: 0.1586 - val_loss: 2.0497
Epoch 2/200
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.2012 - loss: 1.9344 - val_accuracy: 0.1793 - val_loss: 1.9862
Epoch 3/200
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.2863 - loss: 1.7610 - val_accuracy: 0.2207 - val_loss: 1.9353
Epoch 4/200
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.3335 - loss: 1.6415 - val_accuracy: 0.2414 - val_loss: 1.8942
Epoch 5/200
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.3216 - loss: 1.5792 - val_accuracy: 0.2621 - val_loss: 1.8557
Epoch 6/200
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.4158 - loss: 1.4439 - val_accuracy: 0.2828 - val_loss: 1.8188
Epoch 7/200
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.4842 - loss: 1.3762 - val_accuracy: 0.3448 - val_loss: 1.7728
Epoch 8/200
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.4695 - loss: 1.4157 - val_accuracy: 0.

In [67]:
loss, acc = cnn_model.evaluate(x_test, y_test, verbose=0)
print(f"Test Accuracy: {acc * 100:.2f}%")

Test Accuracy: 47.67%


## RNN

In [ ]:
rnn_model = Sequential([
        Input(shape=(max_len, 768)),
        
        SimpleRNN(64, activation= 'tanh'),
        Dropout(0.3),
        
        Dense(32, activation='relu'),
        Dropout(0.4),
        
        Dense(6, activation='softmax')
    ])

rnn_model.compile(loss='categorical_crossentropy', optimizer=Adam(learning_rate=1e-4), metrics=['accuracy'])

rnn_model.summary()

Model: "sequential_23"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_14 (SimpleRNN)       │ (None, 64)             │        53,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_73 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_60 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_74 (Dropout)            │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_61 (Dense)                │ (None, 6)              │           198 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 55,590 (217.15 KB)

 Trainable params: 55,590 (217.15 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
rnn_model.fit(x_train, y_train, epochs=200, batch_size = 32, validation_split=0.2)

Epoch 1/200
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.1742 - loss: 2.0013 - val_accuracy: 0.1379 - val_loss: 1.8022
Epoch 2/200
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.2148 - loss: 1.8164 - val_accuracy: 0.1310 - val_loss: 1.8132
Epoch 3/200
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.2556 - loss: 1.7911 - val_accuracy: 0.1241 - val_loss: 1.8224
Epoch 4/200
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.3210 - loss: 1.7202 - val_accuracy: 0.1103 - val_loss: 1.8195
Epoch 5/200
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.2702 - loss: 1.7229 - val_accuracy: 0.1241 - val_loss: 1.8115
Epoch 6/200
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.3147 - loss: 1.6878 - val_accuracy: 0.1586 - val_loss: 1.8029
Epoch 7/200
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.3227 - loss: 1.6672 - val_accuracy: 0.1310 - val_loss: 1.7914
Epoch 8/200
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.3182 - loss: 1.6423 - val_accuracy: 0.

In [70]:
loss, acc = rnn_model.evaluate(x_test, y_test, verbose=0)
print(f"Test Accuracy: {acc * 100:.2f}%")

Test Accuracy: 42.00%


## LSTM

In [ ]:
lstm_model = Sequential([
    Input(shape=(max_len, 768)),
    Bidirectional(LSTM(128, return_sequences=True)),
    Dropout(0.2),

    Bidirectional(LSTM(64, return_sequences=True)), 
    GlobalAveragePooling1D(),                         
    Dropout(0.2),

    Dense(128, activation='relu'),
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dropout(0.3),

    Dense(6, activation='softmax')
])


lstm_model.compile(loss='categorical_crossentropy', optimizer=Adam(learning_rate=1e-4), metrics=['accuracy'])

lstm_model.summary()

Model: "sequential_24"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional_24                │ (None, 84, 256)        │       918,528 │
│ (Bidirectional)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_75 (Dropout)            │ (None, 84, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_25                │ (None, 84, 128)        │       164,352 │
│ (Bidirectional)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_8      │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_76 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_62 (Dense)                │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_77 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_63 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_78 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_64 (Dense)                │ (None, 6)              │           390 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,108,038 (4.23 MB)

 Trainable params: 1,108,038 (4.23 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
lstm_model.fit(x_train, y_train, epochs=200, batch_size = 32, validation_split=0.2)

Epoch 1/200
19/19 ━━━━━━━━━━━━━━━━━━━━ 4s 121ms/step - accuracy: 0.2032 - loss: 1.7859 - val_accuracy: 0.1586 - val_loss: 1.8192
Epoch 2/200
19/19 ━━━━━━━━━━━━━━━━━━━━ 2s 116ms/step - accuracy: 0.2800 - loss: 1.7243 - val_accuracy: 0.1379 - val_loss: 1.7976
Epoch 3/200
19/19 ━━━━━━━━━━━━━━━━━━━━ 2s 107ms/step - accuracy: 0.3191 - loss: 1.6828 - val_accuracy: 0.1517 - val_loss: 1.7759
Epoch 4/200
19/19 ━━━━━━━━━━━━━━━━━━━━ 2s 110ms/step - accuracy: 0.3190 - loss: 1.6534 - val_accuracy: 0.2138 - val_loss: 1.7606
Epoch 5/200
19/19 ━━━━━━━━━━━━━━━━━━━━ 2s 118ms/step - accuracy: 0.4005 - loss: 1.5745 - val_accuracy: 0.2690 - val_loss: 1.7415
Epoch 6/200
19/19 ━━━━━━━━━━━━━━━━━━━━ 3s 156ms/step - accuracy: 0.4303 - loss: 1.4776 - val_accuracy: 0.2621 - val_loss: 1.7006
Epoch 7/200
19/19 ━━━━━━━━━━━━━━━━━━━━ 3s 165ms/step - accuracy: 0.4625 - loss: 1.4682 - val_accuracy: 0.2483 - val_loss: 1.6889
Epoch 8/200
19/19 ━━━━━━━━━━━━━━━━━━━━ 3s 167ms/step - accuracy: 0.5097 - loss: 1.3653 - val_accu

In [73]:
loss, acc = lstm_model.evaluate(x_test, y_test, verbose=0)
print(f"Test Accuracy: {acc * 100:.2f}%")

Test Accuracy: 40.67%


### Attention + LSTM

In [ ]:
inputs = Input(shape=(max_len, 768), name='inputs')
# return_sequences=True so Attention can attend over time
lstm_out = LSTM(64, activation='tanh', return_sequences=True, name='lstm')(inputs)

# self-attention: query = key = value = lstm_out
attn_out = Attention(use_scale=True, name='self_attention')([lstm_out, lstm_out])

# pool to single vector
context = GlobalAveragePooling1D(name='gap')(attn_out)

h = Dense(32, activation='relu', name='dense_1')(context)
h = Dropout(0.2, name='dropout_1')(h)
outputs = Dense(6, activation='softmax', name='output')(h)

at_lstm_model = Model(inputs=inputs, outputs=outputs, name='simple_attn_lstm')
at_lstm_model.compile(optimizer=Adam(learning_rate=1e-4),
              loss='categorical_crossentropy',
              metrics=['accuracy'])
at_lstm_model.summary()

Model: "simple_attn_lstm"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ inputs (InputLayer) │ (None, 84, 768)   │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ (None, 84, 64)    │    213,248 │ inputs[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ self_attention      │ (None, 84, 64)    │          1 │ lstm[0][0],       │
│ (Attention)         │                   │            │ lstm[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gap                 │ (None, 64)        │          0 │ self_attention[0… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 32)        │      2,080 │ gap[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 32)        │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output (Dense)      │ (None, 6)         │        198 │ dropout_1[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 215,527 (841.90 KB)

 Trainable params: 215,527 (841.90 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
at_lstm_model.fit(x_train, y_train, epochs=200, batch_size = 32, validation_split=0.2)

Epoch 1/200
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 43ms/step - accuracy: 0.2687 - loss: 1.7258 - val_accuracy: 0.1310 - val_loss: 1.8490
Epoch 2/200
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 38ms/step - accuracy: 0.2860 - loss: 1.6448 - val_accuracy: 0.1379 - val_loss: 1.8322
Epoch 3/200
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 35ms/step - accuracy: 0.3330 - loss: 1.5971 - val_accuracy: 0.1793 - val_loss: 1.7711
Epoch 4/200
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 38ms/step - accuracy: 0.3668 - loss: 1.5543 - val_accuracy: 0.1655 - val_loss: 1.7614
Epoch 5/200
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 40ms/step - accuracy: 0.4509 - loss: 1.4619 - val_accuracy: 0.2138 - val_loss: 1.7158
Epoch 6/200
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 41ms/step - accuracy: 0.4214 - loss: 1.4075 - val_accuracy: 0.2276 - val_loss: 1.6962
Epoch 7/200
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 36ms/step - accuracy: 0.4894 - loss: 1.3502 - val_accuracy: 0.2759 - val_loss: 1.6378
Epoch 8/200
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 40ms/step - accuracy: 0.5204 - loss: 1.3061 - val_accuracy: 0.

In [76]:
loss, acc = at_lstm_model.evaluate(x_test, y_test, verbose=0)
print(f"Test Accuracy: {acc * 100:.2f}%")

Test Accuracy: 38.83%
